In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from yellowbrick.cluster import SilhouetteVisualizer
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from pathlib import Path

ModuleNotFoundError: No module named 'yellowbrick'

In [ ]:

# Configuração de visualização
plt.style.use('ggplot')
%matplotlib inline

In [ ]:
# Carrega o dataset
df = pd.read_csv("..\datasets\spotify_processed.csv", encoding='utf-8', low_memory=True)

In [ ]:
CLUSTER_FEATURES = [
    'danceability',
    'energy',
    'loudness',
    'speechiness',
    'acousticness',
    'valence',
    'tempo'
]
X = df[CLUSTER_FEATURES]

In [ ]:

n_clusters_to_test = [5, 6, 7, 11]  # Valores de K que serão testados
n_cols = 2  # Número de colunas na grade de plots
n_rows = int(np.ceil(len(n_clusters_to_test) / n_cols))

fig, axs = plt.subplots(n_rows, n_cols, figsize=(15, 5*n_rows))
axs = axs.ravel()  # Achata o array de eixos para facilitar iteração

for i, k in enumerate(n_clusters_to_test):
    # Cria o visualizador para o K atual
    visualizer = SilhouetteVisualizer(
        KMeans(n_clusters=k, random_state=42, n_init='auto'),
        ax=axs[i],
        colors='yellowbrick'
    )
    
    # Ajusta e plota o visualizador
    visualizer.fit(X)
    visualizer.finalize()
    
    # Configurações adicionais do subplot
    axs[i].set_title(f'K = {k}', fontsize=14)
    axs[i].set_xlabel('Coeficiente de Silhueta')
    axs[i].set_ylabel('Clusters')
    
    # Adiciona linha da silhueta média
    silhouette_score = visualizer.silhouette_score_
    axs[i].axvline(x=silhouette_score, color='red', linestyle='--', 
                   label=f'Média: {silhouette_score:.2f}')
    axs[i].legend()

# Remove eixos vazios se houver
for j in range(i+1, len(axs)):
    fig.delaxes(axs[j])

plt.tight_layout()
plt.suptitle('Análise de Silhueta para Diferentes Valores de K', y=1.02, fontsize=16)
plt.show()

# Salva a figura completa
output_dir = Path("../visualizations")
output_dir.mkdir(exist_ok=True)
fig.savefig(output_dir/'silhouette_analysis.png', dpi=300, bbox_inches='tight')
print(f"Gráficos salvos em: {output_dir/'silhouette_analysis.png'}")